<a href="https://colab.research.google.com/github/1900690/fisheye-distortion-correction/blob/main/%E9%AD%9A%E7%9C%BC%E8%A3%9C%E6%AD%A3%E3%83%BB%E5%8B%95%E7%94%BB%E3%81%8B%E3%82%89%E7%94%BB%E5%83%8F%E3%81%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. AVI動画のアップロード
import os
import cv2
import matplotlib.pyplot as plt
from google.colab import files

# フォルダの初期化（以前のデータをクリア）
for dir_name in ['/content/extracted_images', '/content/corrected_images']:
    !rm -rf {dir_name}
    os.makedirs(dir_name, exist_ok=True)

print("AVI動画ファイルを選択してください。")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"アップロード完了: {video_filename}")

    # 動画の最初のフレームを表示
    cap = cv2.VideoCapture(video_filename)
    if cap.isOpened():
        ret, first_frame = cap.read()
        if ret:
            print("\n【動画の最初のフレーム（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
        else:
            print("動画の最初のフレームを読み込めませんでした。")
        cap.release()
    else:
        print("動画ファイルを開けませんでした。")
else:
    print("アップロードがキャンセルされました。")

In [ ]:
#@title 2. 特定タイミングの画像抽出
import cv2
import os
import matplotlib.pyplot as plt

#@markdown 抽出の条件を指定してください。
撮影開始時間_動画先頭からの経過時間_時間単位 = 1.0 #@param {type:"number"}
抽出するフレーム数_枚数 = 1 #@param {type:"integer"}
切り取り間隔の時間_分単位 = 60 #@param {type:"number"}

if 'video_filename' not in locals():
    print("エラー: セル1で動画をアップロードしてください。")
else:
    # 抽出先フォルダを空にする
    !rm -rf /content/extracted_images/*

    cap = cv2.VideoCapture(video_filename)
    if not cap.isOpened():
        print("エラー: 動画を開けませんでした。")
    else:
        original_fps = cap.get(cv2.CAP_PROP_FPS)
        total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"動画の元のFPS: {original_fps}")
        print(f"動画の総フレーム数: {total_video_frames}")

        # 開始フレーム位置と、指定間隔（分）ごとのフレーム数を計算
        start_frame = int(撮影開始時間_動画先頭からの経過時間_時間単位 * 3600 * original_fps)
        frame_interval = int(切り取り間隔の時間_分単位 * 60 * original_fps)

        if start_frame >= total_video_frames:
            print(f"エラー: 指定された撮影開始時間（{撮影開始時間_動画先頭からの経過時間_時間単位}時間後）は動画の長さを超えています。")
        else:
            extracted_count = 0
            first_image = None

            # 指定されたフレーム数分、間隔をあけて抽出
            for i in range(抽出するフレーム数_枚数):
                target_frame = start_frame + (i * frame_interval)

                # 動画の最大フレーム数を超えたら終了
                if target_frame >= total_video_frames:
                    print(f"警告: 指定されたフレーム数に達する前に動画の終端に達しました。({extracted_count}枚抽出済)")
                    break

                cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
                ret, frame = cap.read()
                if not ret:
                    break

                out_path = f"/content/extracted_images/frame_{target_frame:06d}.jpg"
                cv2.imwrite(out_path, frame)

                if first_image is None:
                    first_image = frame.copy()
                extracted_count += 1

            cap.release()
            print(f"抽出完了: 計 {extracted_count} 枚の画像を保存しました。")

            # 切り取り確認用に抽出した1枚目を表示
            if first_image is not None:
                print("\n【抽出画像の1枚目（切り取り確認用）】")
                plt.figure(figsize=(8, 6))
                plt.imshow(cv2.cvtColor(first_image, cv2.COLOR_BGR2RGB))
                plt.axis('off')
                plt.show()

In [ ]:
#@title 3. 画像の魚眼補正（特定の機種のみ・必要な人のみ実行）
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

#@markdown **魚眼レンズ 補正パラメータ設定**
#@markdown * ※値はPythonの数式やリストの形式（例: [a, b, c]）で入力してください。
DIM_input = "(640, 480)" #@param {type:"string"}
K_input = "[[500.0, 0.0, 320.0], [0.0, 500.0, 240.0], [0.0, 0.0, 1.0]]" #@param {type:"string"}
D_input = "[-0.1, 0.01, 0.0, 0.0]" #@param {type:"string"}

!rm -rf /content/corrected_images/*
extracted_files = sorted(glob.glob('/content/extracted_images/*.jpg'))

if len(extracted_files) > 0:
    print("魚眼補正を実行しています...")
    try:
        # 入力文字列をPythonオブジェクトおよびnumpy配列に変換
        DIM = eval(DIM_input)  # (width, height) のタプル
        K = np.array(eval(K_input), dtype=np.float64)  # 3x3 カメラ行列
        D = np.array(eval(D_input), dtype=np.float64)  # 歪み係数

        # 補正マップの初期化 (fisheyeモデル用)
        map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, DIM, cv2.CV_16SC2)
        first_corrected = None

        for file_path in extracted_files:
            img = cv2.imread(file_path)

            # 画像サイズがDIMと異なる場合はDIMにリサイズ
            if (img.shape[1], img.shape[0]) != DIM:
                img = cv2.resize(img, DIM)

            # 魚眼補正の適用
            undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

            out_path = os.path.join('/content/corrected_images', os.path.basename(file_path))
            cv2.imwrite(out_path, undistorted_img)

            if first_corrected is None:
                first_corrected = undistorted_img.copy()

        print(f"補正完了: {len(extracted_files)} 枚の画像を処理しました。")

        if first_corrected is not None:
            print("\n【補正画像の1枚目（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_corrected, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()

    except Exception as e:
        print(f"エラー: パラメータの解析または補正処理に失敗しました。入力形式を確認してください。\n詳細: {e}")
else:
    print("エラー: 補正する画像がありません。先にセル2を実行して画像を抽出してください。")

In [ ]:
#@title 4. 画像のまとめてダウンロード
import shutil
from google.colab import files
import os

#@markdown ダウンロードするZIPファイル名を指定してください。
zip_filename = "extracted_images" #@param {type:"string"}

corrected_dir = '/content/corrected_images'
extracted_dir = '/content/extracted_images'

# セル3で補正が実行され、画像が生成されているか判定
if os.path.exists(corrected_dir) and len(os.listdir(corrected_dir)) > 0:
    target_dir = corrected_dir
    print("魚眼補正済みの画像を圧縮しています...")
else:
    target_dir = extracted_dir
    print("抽出された元画像を圧縮しています（魚眼補正はスキップされました）...")

if not os.path.exists(target_dir) or len(os.listdir(target_dir)) == 0:
    print("エラー: ダウンロードする画像がありません。")
else:
    # フォルダをZIP化
    zip_path = f"/content/{zip_filename}"
    shutil.make_archive(zip_path, 'zip', root_dir=target_dir)

    # ZIPファイルをダウンロード
    print("ダウンロードを開始します...")
    files.download(f"{zip_path}.zip")

#AVIファイルをmp4に変換する

In [ ]:
#libraryインポート
!pip install opencv-python
!pip install opencv-contrib-python

#AVIからmp4に変換
import cv2
import os

#この中の動画が画像変換される
file_path='/content/AVI/'

#mp4フォルダを作成
if not os.path.exists('/content/mp4'):
    os.mkdir('/content/mp4')

def convert_avi_to_mp4(file_path):
    #VideoCaptureオブジェクトを取得
    cap = cv2.VideoCapture(file_path)

    #動画のプロパティを取得
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    #書き出し設定
    fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
    writer = cv2.VideoWriter("/content/mp4/"+os.path.splitext(os.path.basename(file_path))[0]+".mp4", fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        writer.write(frame)
        if not ret:
            break

    writer.release()
    cap.release()


load_name = os.listdir(file_path)

for file_name in load_name:
  convert_avi_to_mp4(file_path+file_name)

#mp4からjpgを取り出す

In [ ]:
import shutil
import cv2
import os

#この中の動画が画像取り出しされる
LOAD_FOLDA = '/content/mp4'

if os.path.exists("/content/slice"):
  shutil.rmtree("/content/slice")

def save_frame_range(video_path,
                     dir_path, ext='jpg'):

    basename=video_path[-12:-4]
    cap = cv2.VideoCapture(video_path)

    # 総フレーム数
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT));
    #print(total_frames)

    #真ん中+1のフレームを取り出すプラスの数値を調整することでフレームをずらせる
    center_frame=total_frames
    start_frame=0#center_frame-1
    stop_frame=total_frames
    step_frame=1#center_frame

    if not cap.isOpened():
        return

    os.makedirs(dir_path, exist_ok=True)
    base_path = os.path.join(dir_path, basename)

    digit = len(str(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))))

    for n in range(start_frame, stop_frame, step_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, n)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite('{}_{}.{}'.format(base_path, str(n).zfill(digit), ext), frame)
        else:
            return

#スライスフォルダを作成
SAVE_NAME = 'slice'
if not os.path.exists('/content/' +SAVE_NAME):
    os.mkdir('/content/' +SAVE_NAME)

LOAD_NAME = os.listdir(LOAD_FOLDA)

for file_name in LOAD_NAME:
  IMAGE_PATH= LOAD_FOLDA +'/'+file_name
  save_name='/content/'+SAVE_NAME+"/"+os.path.splitext(os.path.basename(file_name))[0]
  os.makedirs(save_name, exist_ok=True)
  save_frame_range(IMAGE_PATH,save_name)

In [ ]:
#画像をまとめてダウンロードするとき用
from google.colab import files
import shutil

shutil.make_archive('slice', format='zip', root_dir='/content/slice/')

files.download('slice.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#魚眼補正

###参考
[魚眼レンズの補正](https://qiita.com/hiro_o_tama/items/cb544fe64ca373750aae)


[OpenCVのundistort（レンズ歪み補正）で端っこが欠けてしまうのをなんとかする](https://qiita.com/jellied_unagi/items/36796d48d7d8a5fb3e42)


[端っこをかけるのを防ぐために](http://twinklesmile.blog42.fc2.com/blog-entry-466.html)

In [ ]:
import cv2
# assert cv2.__version__[0] == '3', 'このモジュールには3.0.0以上の opencv のバージョンが必要です'
import numpy as np
import os
import glob

#チェッカーボードの画像が入っているファイルのまでのパス、画像サイズはすべて同じである必要あり
images = glob.glob('/content/checkerboard/*.jpg')# source/*.jpg

#チェッカーボードのマス数（0から始まる）
CHECKERBOARD = (6,9)

#print(CHECKERBOARD)
subpix_criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_CHECK_COND+cv2.fisheye.CALIB_FIX_SKEW
objp = np.zeros((1, CHECKERBOARD[0]*CHECKERBOARD[1], 3), np.float32)
objp[0,:,:2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
_img_shape = None
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

for fname in images:
    img = cv2.imread(fname)
    if _img_shape == None:
        _img_shape = img.shape[:2]
    else:
        assert _img_shape == img.shape[:2], "All images must share the same size."
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, cv2.CALIB_CB_ADAPTIVE_THRESH+cv2.CALIB_CB_FAST_CHECK+cv2.CALIB_CB_NORMALIZE_IMAGE)
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
        cv2.cornerSubPix(gray,corners,(3,3),(-1,-1),subpix_criteria)
        imgpoints.append(corners)

#print('img end')
N_OK = len(objpoints)
K = np.zeros((3, 3))
D = np.zeros((4, 1))
rvecs = [np.zeros((1, 1, 3), dtype=np.float64) for i in range(N_OK)]
tvecs = [np.zeros((1, 1, 3), dtype=np.float64) for i in range(N_OK)]
rms, _, _, _, _ = \
    cv2.fisheye.calibrate(
        objpoints,
        imgpoints,
        gray.shape[::-1],
        K,
        D,
        rvecs,
        tvecs,
        calibration_flags,
        (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
    )
print("校正に有効な画像が" + str(N_OK) + "枚見つかりました")
print("DIM=" + str(_img_shape[::-1]))
print("K=np.array(" + str(K.tolist()) + ")")
print("D=np.array(" + str(D.tolist()) + ")")

校正に有効な画像が6枚見つかりました
DIM=(1280, 720)
K=np.array([[762.6644982321056, 0.0, 629.5851347102487], [0.0, 762.1060624048673, 363.7359938522793], [0.0, 0.0, 1.0]])
D=np.array([[-0.05837706100996188], [-0.08995973787113537], [0.0976750259892906], [-0.005236625728183422]])


In [ ]:
#指定した画像を構成値を用いて補正
from google.colab.patches import cv2_imshow
import numpy as np
import cv2
import sys
import os

#補正する画像のフォルダpath
load_path="/content/slice/"

if os.path.exists('/content/adjusted/'):
  shutil.rmtree('/content/adjusted/')

# 以下の3行を上のセルの出力に置き換えてください。
DIM=(1280, 720)
K=np.array([[762.6644982321056, 0.0, 629.5851347102487], [0.0, 762.1060624048673, 363.7359938522793], [0.0, 0.0, 1.0]])
D=np.array([[-0.05837706100996188], [-0.08995973787113537], [0.0976750259892906], [-0.005236625728183422]])

#そのままのKだと大きく範囲を切り取ってしまうのでnkを使って補正
nK = K.copy()
nK[0,0]=K[0,0]/1.2
nK[1,1]=K[1,1]/1.2

#補正後の画像を入れるフォルダを作成
if not os.path.exists('/content/adjusted/'):
    os.mkdir('/content/adjusted/')

def undistort(img_path,p):
    img = cv2.imread(img_path)
    h,w = img.shape[:2]
    #map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, DIM, cv2.CV_16SC2)
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), nK, DIM, cv2.CV_16SC2)
    undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    #cv2_imshow(undistorted_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    cv2.imwrite('/content/adjusted/'+p+'/out_' + os.path.basename(img_path), undistorted_img)

if __name__ == '__main__':
    load_name = os.listdir(load_path)
    for p in load_name:
      os.makedirs('/content/adjusted/'+p, exist_ok=True)
      mp4_path=load_path+p
      img_name=os.listdir(mp4_path)
      for i in img_name:
        img_path=mp4_path+'/'+i
        undistort(img_path,p)

In [ ]:
#画像をまとめてダウンロードするとき用
from google.colab import files
import shutil

shutil.make_archive('adjusted', format='zip', root_dir='/content/adjusted/')

files.download('adjusted.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#jpgからmp4を作成

In [ ]:
import cv2
from glob import glob

#動画を作る画像のフォルダpath
load_path="/content/adjusted/"

if os.path.exists('/content/mp4_out/'):
  shutil.rmtree('/content/mp4_out/')

#補正後の動画を入れるフォルダを作成
if not os.path.exists('/content/mp4_out/'):
    os.mkdir('/content/mp4_out/')

# FPS、1秒間の動画で見せる静止画の枚数
FRAME_RATE = 10.0

def timelaps(img_paths):
    # 一枚目の画像サイズを取得
    height, width = cv2.imread(img_paths[0]).shape[:2]

    fourcc = cv2.VideoWriter_fourcc('m','p','4','v')

    video = cv2.VideoWriter('/content/mp4_out/'+os.path.splitext(os.path.basename(img_paths[0]))[0]+'.mp4', fourcc, FRAME_RATE, (width, height))

    for i in range(len(img_paths)):
        img = cv2.imread(img_paths[i])
        # imreadの戻り値に対してNoneじゃないかどうかチェック
        if not img is None:
            img = cv2.resize(img, (width, height))
            video.write(img)

    video.release()

if __name__ == '__main__':

    # imgフォルダ内のファイル一覧取得
    imgs = glob(load_path+'**/*.jpg', recursive=True)
    # 念のため、jpegも取得する
    #imgs.extend(glob('img/*.jpeg'))
    # 画像名称を昇順にソートする
    img_paths = sorted(imgs)
    # 取得画像の枚数を確認する
    #print("Frame枚数{0}".format(len(img_paths)))
    # タイムラプス動画を作成開始
    timelaps(img_paths)

IndexError: list index out of range